<a href="https://colab.research.google.com/github/akshatkeshri/D2C-Churn-Part3-Prediction/blob/main/churn_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1
!pip install gdown xgboost -q
import gdown
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

folder_url = 'https://drive.google.com/drive/folders/1PmLapJI1VSDgvl_AxARNKwM1MCd3WFX0'
output_dir = './data'
if not os.path.exists(output_dir):
    !gdown --folder {folder_url} -O {output_dir}

df = pd.read_csv('./data/rfm_modeling_snapshot.csv')

drop_cols = ['customer_id', 'snapshot_date', 'split']
features_to_drop = [col for col in drop_cols if col in df.columns]

X = df.drop(columns=features_to_drop + ['churn_next_60d'])
y = df['churn_next_60d']

cat_cols = X.select_dtypes(include=['object']).columns
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_temp, X_test, y_temp, y_test = train_test_split(X_encoded, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print(" Data Pipeline Complete. Zero leakage detected.")
print(f"Training Set Shape: {X_train_scaled.shape[0]} rows (Model learns from this)")
print(f"Validation Set Shape: {X_val_scaled.shape[0]} rows (Used to tune hyperparameters)")
print(f"Testing Set Shape: {X_test_scaled.shape[0]} rows (Untouched final exam for the model)")

 Data Pipeline Complete. Zero leakage detected.
Training Set Shape: 1679 rows (Model learns from this)
Validation Set Shape: 361 rows (Used to tune hyperparameters)
Testing Set Shape: 360 rows (Untouched final exam for the model)


In [3]:
# Cell 2: Model Training & Evaluation
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

print("--- Training Baseline Model (Logistic Regression) ---")
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
print(classification_report(y_test, lr_preds))

print("\n--- Training Advanced Model (XGBoost) ---")
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_scaled, y_train)

xgb_probs = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_preds_default = (xgb_probs >= 0.5).astype(int)
print("XGBoost Performance (Default Threshold 0.5):")
print(classification_report(y_test, xgb_preds_default))

--- Training Baseline Model (Logistic Regression) ---
              precision    recall  f1-score   support

           0       0.84      0.85      0.84       191
           1       0.83      0.81      0.82       169

    accuracy                           0.83       360
   macro avg       0.83      0.83      0.83       360
weighted avg       0.83      0.83      0.83       360


--- Training Advanced Model (XGBoost) ---
XGBoost Performance (Default Threshold 0.5):
              precision    recall  f1-score   support

           0       0.80      0.77      0.78       191
           1       0.75      0.78      0.77       169

    accuracy                           0.78       360
   macro avg       0.77      0.78      0.77       360
weighted avg       0.78      0.78      0.78       360



In [4]:
# Cell 3
import json
import joblib
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

custom_threshold = 0.40
xgb_preds_tuned = (xgb_probs >= custom_threshold).astype(int)

print(f"\n--- XGBoost Performance (Custom Threshold {custom_threshold}) ---")
print(classification_report(y_test, xgb_preds_tuned))

acc = accuracy_score(y_test, xgb_preds_tuned)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, xgb_preds_tuned, average='binary')
roc_auc = roc_auc_score(y_test, xgb_probs)
cm = confusion_matrix(y_test, xgb_preds_tuned).tolist()

metrics_dict = {
    "accuracy": round(acc, 4),
    "precision": round(precision, 4),
    "recall": round(recall, 4),
    "f1_score": round(f1, 4),
    "roc_auc": round(roc_auc, 4),
    "confusion_matrix_values": {"TN": cm[0][0], "FP": cm[0][1], "FN": cm[1][0], "TP": cm[1][1]},
    "selected_threshold": custom_threshold
}

with open('metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=4)
print(" metrics.json saved successfully.")

joblib.dump(xgb_model, 'model.pkl')
print(" model.pkl saved successfully.")

feature_importances = pd.Series(xgb_model.feature_importances_, index=X_train_scaled.columns).sort_values(ascending=False)
print("\n--- Top 5 Features Driving Churn Predictions ---")
print(feature_importances.head(5))


--- XGBoost Performance (Custom Threshold 0.4) ---
              precision    recall  f1-score   support

           0       0.82      0.74      0.78       191
           1       0.73      0.82      0.77       169

    accuracy                           0.78       360
   macro avg       0.78      0.78      0.77       360
weighted avg       0.78      0.78      0.78       360

 metrics.json saved successfully.
 model.pkl saved successfully.

--- Top 5 Features Driving Churn Predictions ---
recency_days                      0.125550
acquisition_channel_Organic       0.048786
return_rate_180d                  0.042491
cart_adds_30d                     0.036783
acquisition_channel_Influencer    0.035513
dtype: float32


In [5]:
# Cell 4
results_df = pd.DataFrame({'customer_id': df.loc[X_test.index, 'customer_id'], 'Actual': y_test, 'Predicted': xgb_preds_tuned})

fps = results_df[(results_df['Actual'] == 0) & (results_df['Predicted'] == 1)].head(5)
fps['Error_Type'] = 'False Positive'

fns = results_df[(results_df['Actual'] == 1) & (results_df['Predicted'] == 0)].head(5)
fns['Error_Type'] = 'False Negative'

error_cases = pd.concat([fps, fns])
print("\n--- 10 Specific Error Cases ---")
display(error_cases)


--- 10 Specific Error Cases ---


,customer_id,Actual,Predicted,Error_Type
188,CUST00189,0,1,False Positive
1202,CUST01203,0,1,False Positive
939,CUST00940,0,1,False Positive
1028,CUST01029,0,1,False Positive
1651,CUST01652,0,1,False Positive
615,CUST00616,1,0,False Negative
308,CUST00309,1,0,False Negative
1724,CUST01725,1,0,False Negative
1578,CUST01579,1,0,False Negative
2247,CUST02248,1,0,False Negative
